# Fine-tuning GPT-2 para Generación de Contratos de Alquiler

**Modelo Utilizado:** `distilgpt2`  
**Tarea:** Generación de texto (Text-to-Text) a partir de parámetros estructurados  
**Tipo de modelo:** Decoder (modelo generativo autoregresivo)

---

El modelo recibe como entrada un conjunto de parámetros clave del contrato:

- ARRENDADOR  
- ARRENDATARIO  
- DIRECCION  
- RENTA  
- FECHA_INICIO  
- DURACION  

Y genera como salida un texto coherente con formato semi-formal legal.

## Hacemos las instalaciones necesarias

In [ ]:
!pip install transformers datasets pandas -q

## Importaciones

In [1]:
import pandas as pd
import json
from transformers import GPT2Tokenizer, GPT2LMHeadModel, Trainer, TrainingArguments
from datasets import Dataset
from google.colab import files
from sklearn.model_selection import train_test_split

## Cargamos el dataset

In [2]:
uploaded = files.upload()

df = pd.read_json(list(uploaded.keys())[0])

df.head()

Saving contratos_ner.json to contratos_ner (2).json


,tokens,ner_tags
0,"[Juan, Pérez, arrienda, a, María, López, el, p...","[B-ARRENDADOR, I-ARRENDADOR, O, O, B-ARRENDATA..."
1,"[Carlos, Ruiz, como, arrendador, cede, a, Ana,...","[B-ARRENDADOR, I-ARRENDADOR, O, O, O, O, B-ARR..."
2,"[Pedro, Sánchez, arrienda, a, Laura, Martínez,...","[B-ARRENDADOR, I-ARRENDADOR, O, O, B-ARRENDATA..."
3,"[Luis, Fernández, cede, en, arrendamiento, a, ...","[B-ARRENDADOR, I-ARRENDADOR, O, O, O, O, B-ARR..."
4,"[Antonio, Mora, arrienda, a, Elena, Castro, el...","[B-ARRENDADOR, I-ARRENDADOR, O, O, B-ARRENDATA..."


## Convertimos tokens y tags en un diccionario estructurado

In [3]:
def extract_entities(tokens, tags):
    entities = {}
    current_entity = None
    current_tokens = []

    for token, tag in zip(tokens, tags):
        if tag.startswith("B-"):
            if current_entity:
                entities[current_entity] = " ".join(current_tokens)
            current_entity = tag[2:]
            current_tokens = [token]

        elif tag.startswith("I-") and current_entity:
            current_tokens.append(token)

        else:
            if current_entity:
                entities[current_entity] = " ".join(current_tokens)
                current_entity = None
                current_tokens = []

    if current_entity:
        entities[current_entity] = " ".join(current_tokens)

    return entities

## Le aplicamos la función a todo el dataset

In [4]:
data = []

for _, row in df.iterrows():
    tokens = row["tokens"]
    tags = row["ner_tags"]

    text = " ".join(tokens)
    entities = extract_entities(tokens, tags)

    data.append({
        "text": text,
        "entities": entities
    })

structured_df = pd.DataFrame(data)
structured_df.head()

,text,entities
0,Juan Pérez arrienda a María López el piso en C...,"{'ARRENDADOR': 'Juan Pérez', 'ARRENDATARIO': '..."
1,Carlos Ruiz como arrendador cede a Ana García ...,"{'ARRENDADOR': 'Carlos Ruiz', 'ARRENDATARIO': ..."
2,Pedro Sánchez arrienda a Laura Martínez el loc...,"{'ARRENDADOR': 'Pedro Sánchez', 'ARRENDATARIO'..."
3,Luis Fernández cede en arrendamiento a Rosa To...,"{'ARRENDADOR': 'Luis Fernández', 'ARRENDATARIO..."
4,Antonio Mora arrienda a Elena Castro el aparta...,"{'ARRENDADOR': 'Antonio Mora', 'ARRENDATARIO':..."


## Creamos la plantilla general que servirá como contrato y que el modelo deberá aprender

In [5]:
def generate_contract(e):
    return f"""En Madrid, Juan Pérez arrienda a María López la vivienda en Calle Mayor 5 por 800 € durante 12 meses desde el 1 de enero de 2024."""

## Construimos el imput que el modelo espera

In [6]:
def create_prompt(e):
    return f"""Genera un contrato de alquiler con los siguientes datos:

arrendador: {e.get('ARRENDADOR','')}
arrendatario: {e.get('ARRENDATARIO','')}
direccion: {e.get('DIRECCION','')}
renta: {e.get('RENTA','')}
fecha_inicio: {e.get('FECHA_INICIO','')}
duracion: {e.get('DURACION','')}

Contrato:
"""

## Creamos pares listos para entrenamiento

In [7]:
inputs = []
outputs = []

for _, row in structured_df.iterrows():
    e = row["entities"]

    prompt = create_prompt(e)
    contract = generate_contract(e)

    inputs.append(prompt)
    outputs.append(contract)

final_df = pd.DataFrame({
    "input": inputs,
    "output": outputs
})

final_df.head()

,input,output
0,Genera un contrato de alquiler con los siguien...,"En Madrid, Juan Pérez arrienda a María López l..."
1,Genera un contrato de alquiler con los siguien...,"En Madrid, Juan Pérez arrienda a María López l..."
2,Genera un contrato de alquiler con los siguien...,"En Madrid, Juan Pérez arrienda a María López l..."
3,Genera un contrato de alquiler con los siguien...,"En Madrid, Juan Pérez arrienda a María López l..."
4,Genera un contrato de alquiler con los siguien...,"En Madrid, Juan Pérez arrienda a María López l..."


## Hacemos la división train / test

In [8]:
train_df, test_df = train_test_split(final_df, test_size=0.2, random_state=42)

print("Train:", len(train_df))
print("Test:", len(test_df))

Train: 16
Test: 4


## Convertimos a Hugging Face, necesario para entrenar

In [9]:
train_dataset = Dataset.from_pandas(train_df)
test_dataset = Dataset.from_pandas(test_df)
train_dataset

Dataset({
    features: ['input', 'output', '__index_level_0__'],
    num_rows: 16
})

## Modelo Generativo (Decoder)

**Modelo base elegido:** `distilgpt2`

---

### ¿Por qué este modelo?

Se ha seleccionado `distilgpt2`, una versión reducida de GPT-2, por las siguientes razones:

- Es un modelo generativo autoregresivo (decoder), adecuado para tareas de generación de texto.
- Tiene un tamaño reducido, lo que permite entrenarlo en entornos con recursos limitados como Google Colab.
- Mantiene un buen equilibrio entre rendimiento y eficiencia, siendo suficiente para demostrar el flujo completo del proyecto.

---

### Enfoque del problema

El modelo se entrena para resolver una tarea de tipo **Text-to-Text**, donde:

- **Entrada:** parámetros estructurados del contrato (arrendador, arrendatario, renta, etc.)
- **Salida:** texto de contrato en lenguaje natural

Este enfoque sigue el caso de uso definido en el enunciado del proyecto, donde se busca generar borradores de contratos a partir de información previamente extraída mediante NER.

---

### Limitaciones del enfoque

- El modelo base está preentrenado mayoritariamente en inglés, lo que puede provocar mezcla de idiomas en la generación.
- El dataset disponible es reducido, lo que limita la capacidad de generalización del modelo.
- El modelo puede tender a repetir partes del prompt o generar contenido incoherente.

Estas limitaciones se tendrán en cuenta en la evaluación cualitativa.

## El tokenizer usando distilgpt2, GPT-2 optimizado para colab

In [10]:
tokenizer = GPT2Tokenizer.from_pretrained("distilgpt2")
tokenizer.pad_token = tokenizer.eos_token

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


## Convertimos el texto en tokens

In [11]:
def tokenize(example):
    prompt = example["input"]
    output = example["output"]

    full_text = prompt + output

    # Tokenizar sin padding
    full_enc = tokenizer(full_text, truncation=True, max_length=256)
    prompt_enc = tokenizer(prompt, truncation=True, max_length=256)

    input_ids = full_enc["input_ids"]
    attention_mask = full_enc["attention_mask"]

    labels = input_ids.copy()

    # Ignorar prompt
    prompt_len = len(prompt_enc["input_ids"])
    labels[:prompt_len] = [-100] * prompt_len

    # Padding manual
    pad_length = 256 - len(input_ids)

    input_ids += [tokenizer.pad_token_id] * pad_length
    attention_mask += [0] * pad_length
    labels += [-100] * pad_length

    return {
        "input_ids": input_ids,
        "attention_mask": attention_mask,
        "labels": labels
    }


train_dataset = train_dataset.map(tokenize)
test_dataset = test_dataset.map(tokenize)

Map:   0%|          | 0/16 [00:00<?, ? examples/s]

Map:   0%|          | 0/4 [00:00<?, ? examples/s]

## Cargamos el modelo

In [12]:
model = GPT2LMHeadModel.from_pretrained("distilgpt2")

Loading weights:   0%|          | 0/76 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: distilgpt2
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
transformer.h.{0, 1, 2, 3, 4, 5}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


## Configuramos el entrenamiento

In [13]:
training_args = TrainingArguments(
    output_dir="./results",
    per_device_train_batch_size=2,
    num_train_epochs=10,
    logging_steps=10,
    save_steps=50,
    save_total_limit=2,
    fp16=True
)

## Iniciamos el entrenamiento y entrenamos

In [14]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset
)

trainer.train()

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
`loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.


Step,Training Loss
10,3.126332
20,0.505298
30,0.038918
40,0.005849
50,0.001382
60,0.000765
70,0.000964
80,0.000619


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=80, training_loss=0.46001606663921846, metrics={'train_runtime': 540.1977, 'train_samples_per_second': 0.296, 'train_steps_per_second': 0.148, 'total_flos': 10451870023680.0, 'train_loss': 0.46001606663921846, 'epoch': 10.0})

## Probamos el modelo con un ejemplo

In [15]:
def generate_text(prompt):
    inputs = tokenizer(prompt, return_tensors="pt")

    outputs = model.generate(
        **inputs,
        max_length=180,
        do_sample=True,
        temperature=0.6,
        top_p=0.85,
        repetition_penalty=1.3,
        no_repeat_ngram_size=3
    )

    return tokenizer.decode(outputs[0], skip_special_tokens=True)

## Hacemos una evalución del modelo

In [16]:
pd.set_option('display.max_colwidth', None)

results = []

for i in range(3):
    e = structured_df.iloc[i]["entities"]

    prompt = create_prompt(e)
    expected = generate_contract(e)
    generated = generate_text(prompt)

    # Análisis inicial
    if len(generated) < len(expected) * 0.7:
        analisis = "El modelo genera un texto coherente pero demasiado corto, simplificando la estructura del contrato."
    elif "Madrid" not in generated:
        analisis = "El modelo omite elementos clave como la localización, lo que indica falta de generalización."
    else:
        analisis = "El modelo produce texto incoherente con datos inventados y cambia completamente el contexto del contrato. No respeta el formato ni utiliza correctamente la información proporcionada, lo que refleja limitaciones importantes del modelo."

    results.append({
        "input": prompt,
        "generado": generated,
        "esperado": expected,
        "analisis": analisis
    })

eval_df = pd.DataFrame(results)
eval_df

Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


,input,generado,esperado,analisis
0,Genera un contrato de alquiler con los siguientes datos:\n\narrendador: Juan Pérez\narrendatario: María López\ndireccion: Calle Mayor 5\nrenta: 800 € mensuales\nfecha_inicio: 1 de enero de 2024\nduracion: 12 meses\n\nContrato:\n,"Genera un contrato de alquiler con los siguientes datos:\n\narrendador: Juan Pérez\narrendatario: María López\ndireccion: Calle Mayor 5\nrenta: 800 € mensuales\nfecha_inicio: 1 de enero de 2024\nduracion: 12 meses\n\nContrato:\nEn Madrid, Luis Aragonado arrienda a vivienda en the University of California-San Diego (University of Southern San Francisco) and Universidad Autónoma La Rasa Los Altidores Asícetados Un Washington D.C., as planned by President Donald Trump on September 18th at 10 p… En el 6 por la cuy this week in El Paolo Petruz Performing Arts Center for Science Education – Santa","En Madrid, Juan Pérez arrienda a María López la vivienda en Calle Mayor 5 por 800 € durante 12 meses desde el 1 de enero de 2024.","El modelo produce texto incoherente con datos inventados y cambia completamente el contexto del contrato. No respeta el formato ni utiliza correctamente la información proporcionada, lo que refleja limitaciones importantes del modelo."
1,Genera un contrato de alquiler con los siguientes datos:\n\narrendador: Carlos Ruiz\narrendatario: Ana García\ndireccion: Avenida Constitución 12\nrenta: 650 €\nfecha_inicio: 1 de marzo de 2024\nduracion: 6 meses\n\nContrato:\n,"Genera un contrato de alquiler con los siguientes datos:\n\narrendador: Carlos Ruiz\narrendatario: Ana García\ndireccion: Avenida Constitución 12\nrenta: 650 €\nfecha_inicio: 1 de marzo de 2024\nduracion: 6 meses\n\nContrato:\nEn Madrid, Juan Pérez arrienda a Marían López la vivienda en Calle Mayor 5 por 800 € durante 11 el 10 desdeen El Ningel 2 por 500 euros enero del cual. En Ciudad Oligedigang La Presse Las Palacio 8 emblazoners (Minyuglian WEDT) and an allocation of $1 million to the Los Angeles County Commission on","En Madrid, Juan Pérez arrienda a María López la vivienda en Calle Mayor 5 por 800 € durante 12 meses desde el 1 de enero de 2024.","El modelo produce texto incoherente con datos inventados y cambia completamente el contexto del contrato. No respeta el formato ni utiliza correctamente la información proporcionada, lo que refleja limitaciones importantes del modelo."
2,Genera un contrato de alquiler con los siguientes datos:\n\narrendador: Pedro Sánchez\narrendatario: Laura Martínez\ndireccion: Plaza España 3\nrenta: 1200 €\nfecha_inicio: 15 de febrero de 2024\nduracion: 24 meses\n\nContrato:\n,"Genera un contrato de alquiler con los siguientes datos:\n\narrendador: Pedro Sánchez\narrendatario: Laura Martínez\ndireccion: Plaza España 3\nrenta: 1200 €\nfecha_inicio: 15 de febrero de 2024\nduracion: 24 meses\n\nContrato:\nEn Madrid, Juan Pérez arrienda a María López la vivienda en Calle Mayor 5 por 800 € durante 12 Messe desde el 1 doreen Los Angeles Times obituar one zou el 2 Domeno Cup favorites. En Madrid, Luis Arroyan poder esporte an Arnold Schwarzenegger stadium 7 mesements olvide adidas 13 mesés entordes El Correa le Ver","En Madrid, Juan Pérez arrienda a María López la vivienda en Calle Mayor 5 por 800 € durante 12 meses desde el 1 de enero de 2024.","El modelo produce texto incoherente con datos inventados y cambia completamente el contexto del contrato. No respeta el formato ni utiliza correctamente la información proporcionada, lo que refleja limitaciones importantes del modelo."


## Conclusión del modelo generativo

El modelo `distilgpt2` ha sido capaz de generar texto a partir de parámetros estructurados, siguiendo el enfoque planteado en el proyecto. Tras aplicar mejoras como el enmascaramiento del prompt en la función de pérdida y la simplificación de la estructura del output, se observa una ligera mejora en la coherencia de la generación y en la capacidad del modelo para reproducir patrones similares a contratos.

---

### Aspectos positivos

- El modelo ha aprendido parcialmente la estructura general del contrato.
- Es capaz de iniciar la generación en el formato esperado ("En Madrid, ...").
- Se observa una mejora en la coherencia respecto a versiones anteriores tras aplicar el enmascaramiento del prompt.
- La simplificación del output ha reducido parcialmente el ruido en la generación.
- Se ha implementado correctamente el flujo completo: parámetros → texto generado.

---

### Limitaciones detectadas

- El modelo sigue sin utilizar correctamente los parámetros de entrada, mezclando datos entre ejemplos.
- Continúa generando contenido incoherente o inventado en muchas ocasiones.
- Mezcla idiomas (español e inglés), debido al preentrenamiento del modelo base.
- No mantiene de forma consistente el formato esperado del contrato.
- Aunque se ha reducido, aún existe cierta repetición del prompt en la generación.

---

### Causa de las limitaciones

Estas limitaciones se deben principalmente a:

- El reducido tamaño del dataset de entrenamiento (~20 ejemplos).
- El uso de datos sintéticos basados en plantillas en lugar de contratos reales.
- El uso de un modelo preentrenado en inglés para una tarea en español.
- La limitada capacidad de generalización del modelo con pocos datos.

---

### Posibles mejoras

Para mejorar el rendimiento del modelo en un entorno real se podrían aplicar:

- Aumentar el tamaño y la calidad del dataset (más contratos reales).
- Utilizar un modelo preentrenado en español.
- Aplicar técnicas como RAG (Retrieval-Augmented Generation) para aportar contexto verificado.
- Ajustar aún más el proceso de entrenamiento (fine-tuning más prolongado o técnicas adicionales).
- Afinar los parámetros de generación para mejorar la coherencia del texto.

---

### Conclusión final

El modelo demuestra correctamente el flujo completo de un sistema basado en LLMs y refleja mejoras tras la aplicación de técnicas de entrenamiento como el enmascaramiento del prompt. Sin embargo, sigue presentando limitaciones significativas en la calidad de generación, principalmente debido al tamaño reducido del dataset. Para un uso en producción, sería necesario mejorar tanto los datos como el modelo.

## Guardamos el modelo

In [ ]:
model.save_pretrained("modelo_gpt2_contratos")
tokenizer.save_pretrained("modelo_gpt2_contratos")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('modelo_gpt2_contratos/tokenizer_config.json',
 'modelo_gpt2_contratos/tokenizer.json')